# Mk33: does memory change task success?

The earlier memory notebooks measured **next-token loss**. This one measures what matters for an agent:
**does it solve more of the task?** The Mk33 agent runs end to end on the bench tasks with each memory method, at an
equal memory budget, and the report compares final scores with paired bootstrap 95% intervals.

| mode | how old turns are forgotten |
| --- | --- |
| `truncate` | oldest turns are cut from the text (baseline) |
| `rlc` | old turns are replaced by the round ledger |
| `retrieve` | old turns go into BM25 retrieval memory and relevant ones come back |
| `kv:full` | nothing is forgotten (upper bound, uses 4x the memory) |
| `kv:h2o` | the model keeps the full text; its KV cache is cut to the same budget by H2O eviction |

**Before starting:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all.

**This takes several sessions.** The full grid is 30 agent runs of up to 25 minutes each, longer than one free Colab
session. Results are saved to your Google Drive after every run, so when Colab disconnects, open the notebook again
and Run all: finished runs are skipped. The report at the end works with whatever has finished so far, and runs are
ordered so that each repeat covers every mode before the next repeat starts.

## 1. GPU and model

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run again."
mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {torch.cuda.get_device_name()}, {mem_gb:.0f} GB")

MODEL = "Qwen/Qwen2.5-3B-Instruct"  # the model from the memory study; tool calling works with the HF backend
DTYPE = "float16"                    # T4 has no fast bfloat16

## 2. Save results to Google Drive

Approve the Drive prompt. Everything goes to `My Drive/mk33/task_success/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/mk33/task_success"
import os; os.makedirs(OUT, exist_ok=True)
print("results folder:", OUT)

## 3. Install Mk33

In [ ]:
%cd /content
!test -d Mk33 || git clone -q https://github.com/dulguuntsatsralt-cell/Mk33
%cd /content/Mk33
!git pull -q
!pip install -q -e '.[mk33-kv,dev]'
!git log --oneline -1

## 4. Prepare the tasks and check the pipeline

Task generation is seeded, so every session builds identical environments. The dry run checks the whole pipeline
(workspace copy, scorer, ledger, report) in a few seconds without loading the model.

In [ ]:
!mk33 bench-prepare --out bench/envs --tasks knapsack,regression
!mk33 bench-run --dry-run --tasks knapsack --modes rlc --repeats 1 --max-rounds 2 --out /content/dry --force | tail -2

## 5. Run the grid

Settings shared by every mode:

* 2 tasks (knapsack, regression) x 3 repeats
* up to 8 rounds, 40 tool calls and 25 minutes per run
* memory budget 8,000 tokens: the text window for `truncate`, `rlc` and `retrieve`, and the KV-cache budget for `kv:h2o`
* `kv:full` keeps up to 32,000 tokens and is the no-forgetting reference

H2O needs eager attention, so it runs in its own call with a smaller prefill chunk. Both calls write to the same
folder and the report combines them.

In [ ]:
COMMON = (f"--backend hf --model {MODEL} --dtype {DTYPE} --envs bench/envs/envs.jsonl --out {OUT} "
          "--tasks knapsack,regression --max-rounds 8 --max-tool-calls 40 --max-minutes 25 "
          "--window 8000 --kv-budget 8000 --kv-text-window 32000 --temperature 0.7")

for rep in (1, 2, 3):  # finish repeat 1 for every mode before starting repeat 2
    print(f"=== repeat {rep} ===")
    !mk33 bench-run {COMMON} --repeats {rep} --modes truncate,rlc,retrieve,kv:full 2>&1 | grep -v -i warning
    !mk33 bench-run {COMMON} --repeats {rep} --modes kv:h2o --kv-chunk 128 2>&1 | grep -v -i warning

## 6. Report

`report.md` has the tables: mean best score per mode, the share of runs that beat the reference solution, and paired
differences with 95% intervals. A difference counts as real only when its interval excludes zero.

In [ ]:
!mk33 bench-report --out {OUT}

In [ ]:
import json, statistics
import matplotlib.pyplot as plt

s = json.load(open(f"{OUT}/summary.json"))
order = [m for m in ["truncate", "rlc", "retrieve", "kv:h2o", "kv:full"] if m in s["modes"]]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

means = [statistics.mean(s["per_mode"][m]["best"]) for m in order]
for i, m in enumerate(order):
    xs = s["per_mode"][m]["best"]
    ax1.scatter([i] * len(xs), xs, color="0.6", s=18, zorder=2)
ax1.bar(range(len(order)), means, color=["#888" if m in ("truncate", "kv:full") else "#3b5bdb" for m in order], zorder=1)
ax1.axhline(1.0, ls="--", lw=1, color="k"); ax1.text(len(order) - 0.5, 1.01, "reference", ha="right", fontsize=8)
ax1.set_xticks(range(len(order)), order); ax1.set_ylabel("best normalized score"); ax1.set_title("Final score per run")

for m in order:
    c = s["curves"][m]
    ax2.plot(range(1, len(c) + 1), c, marker="o", ms=3, label=m)
ax2.set_xlabel("round"); ax2.set_ylabel("best so far (mean)"); ax2.set_title("Progress over rounds"); ax2.legend()
fig.tight_layout()
os.makedirs(f"{OUT}/figures", exist_ok=True)
fig.savefig(f"{OUT}/figures/task_success.png", dpi=200)
plt.show()

done = sum(s["per_mode"][m]["n"] for m in order)
print(f"{done} of 30 runs finished ({s['failed_runs']} failed). Rerun the notebook to continue the grid.")

## 7. Download

Everything is already on your Drive. This also downloads a zip of the report, figure and run summaries (the full
trajectories stay on Drive because they are large).

In [ ]:
!cd {OUT} && zip -qr /content/mk33_task_success.zip report.md summary.json curves.csv figures runs -x '*/workspace/*' '*/trajectory.json'
from google.colab import files
files.download("/content/mk33_task_success.zip")